# Parkinson's UPDRS Prediction — XGBoost (Production)

### Key improvements over baseline notebook
| Issue in original | Fix applied here |
|---|---|
| `StandardScaler` + `PCA` fit on full dataset (leakage) | Fit **only on train split**, transform test separately |
| `train_test_split` is recording-wise (leakage) | **Subject-aware split** — all recordings of a subject go to train OR test only |
| `test_time` included in features | Dropped — not available at inference time |
| No early stopping | `early_stopping_rounds=50` added |
| Results stored in loose variables | Collected into a clean `results_df` |
| No reproducibility seed for subject split | Fixed seed + printed test subjects for transparency |

In [ ]:
# ═══════════════════════════════════════════════════════════
# 0. IMPORTS
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. LOAD & BASIC CHECKS
# ═══════════════════════════════════════════════════════════
df = pd.read_csv("../src/data/parkinsons_updrs.csv")

print(f"Shape       : {df.shape}")
print(f"Subjects    : {df['subject#'].nunique()}")
print(f"Missing vals: {df.isnull().sum().sum()}")
df.head()

In [ ]:
df.describe()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CONSTANTS
# ═══════════════════════════════════════════════════════════
TARGET      = "total_UPDRS"
AGE_COL     = "age"
GENDER_COL  = "sex"      # 0 = female, 1 = male (per dataset docs)

# Voice features only — test_time dropped (not available at inference)
VOICE_COLS = [
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer",   "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA",
    "NHR", "HNR", "RPDE", "DFA", "PPE"
]
# age & sex kept outside PCA — see notebook header for rationale
META_COLS   = [AGE_COL, GENDER_COL]
N_PCA_COMPONENTS = 8   # retains ~95% variance; age loads on PC7 → preserved separately

XGB_PARAMS = dict(
    n_estimators        = 1000,    # generous upper bound; early stopping controls actual
    learning_rate       = 0.01,
    max_depth           = 6,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    objective           = "reg:squarederror",
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    early_stopping_rounds = 50,
    eval_metric         = "rmse",
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. AGE GROUPS & GENDER LABELS  (for analysis only, not fed raw to model)
# ═══════════════════════════════════════════════════════════
def assign_age_group(age):
    if age < 60:   return "lt60"
    elif age <= 70: return "60to70"
    else:           return "gt70"

df["age_group"] = df[AGE_COL].apply(assign_age_group)
df["gender"]    = df[GENDER_COL].map({0: "Female", 1: "Male"})
df["group_key"] = df["gender"] + "_" + df["age_group"]

print("Group distribution (recordings):")
print(df.groupby("group_key")["subject#"].agg(["count", "nunique"])
        .rename(columns={"count": "recordings", "nunique": "subjects"}))

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. SUBJECT-AWARE TRAIN / TEST SPLIT
#
#    Split at the SUBJECT level so no subject's recordings
#    appear in both train and test → prevents data leakage.
#
#    Done independently per group_key so each group keeps
#    its own 80/20 ratio of subjects.
# ═══════════════════════════════════════════════════════════
TEST_SUBJECT_RATIO = 0.20

train_subjects_all, test_subjects_all = [], []

for gkey, gdf in df.groupby("group_key"):
    subj = gdf["subject#"].unique()
    np.random.shuffle(subj)
    n_test = max(1, round(len(subj) * TEST_SUBJECT_RATIO))
    test_subjects_all.extend(subj[:n_test])
    train_subjects_all.extend(subj[n_test:])

train_subjects_all = list(set(train_subjects_all))
test_subjects_all  = list(set(test_subjects_all))

# Sanity check — zero overlap
assert len(set(train_subjects_all) & set(test_subjects_all)) == 0, \
    "LEAKAGE: subject appears in both train and test!"

train_df = df[df["subject#"].isin(train_subjects_all)].copy()
test_df  = df[df["subject#"].isin(test_subjects_all)].copy()

print(f"Train: {len(train_subjects_all):>3} subjects | {len(train_df):>5} recordings")
print(f"Test : {len(test_subjects_all):>3} subjects | {len(test_df):>5} recordings")
print(f"Test subjects: {sorted(test_subjects_all)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. PREPROCESSING  (fit on train only — transform test)
#
#    Voice features  → StandardScaler → PCA(8)
#    age, sex        → StandardScaler  (kept separate from PCA)
#    Final features  → [PC1..PC8, age_scaled, sex_scaled] = 10 cols
# ═══════════════════════════════════════════════════════════
scaler_voice = StandardScaler()
scaler_meta  = StandardScaler()
pca          = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_STATE)

# Fit ONLY on training data
X_voice_train = scaler_voice.fit_transform(train_df[VOICE_COLS])
X_voice_test  = scaler_voice.transform(test_df[VOICE_COLS])

X_meta_train  = scaler_meta.fit_transform(train_df[META_COLS])
X_meta_test   = scaler_meta.transform(test_df[META_COLS])

X_pca_train   = pca.fit_transform(X_voice_train)
X_pca_test    = pca.transform(X_voice_test)

# Final feature matrices
X_train = np.hstack([X_pca_train, X_meta_train])
X_test  = np.hstack([X_pca_test,  X_meta_test])

y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values

FEATURE_NAMES = [f"PC{i+1}" for i in range(N_PCA_COMPONENTS)] + META_COLS

print(f"PCA explained variance : {pca.explained_variance_ratio_.sum():.3f}")
print(f"Per component          : {np.round(pca.explained_variance_ratio_, 3)}")
print(f"Final feature shape    : train={X_train.shape}, test={X_test.shape}")

# Eigenvalue table
eigen_df = pd.DataFrame({
    "PC"                      : [f"PC{i+1}" for i in range(N_PCA_COMPONENTS)],
    "Eigenvalue"              : pca.explained_variance_,
    "Explained Variance Ratio": pca.explained_variance_ratio_,
    "Cumulative Variance"     : np.cumsum(pca.explained_variance_ratio_),
})
print("\n", eigen_df.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. CORE TRAINING FUNCTION
# ═══════════════════════════════════════════════════════════
def train_xgb_group(
    X_tr, y_tr,
    X_te, y_te,
    group_name: str,
    verbose: bool = False
):
    """
    Train XGBRegressor on pre-split, pre-scaled data.
    Returns (model, metrics_dict).
    """
    if len(X_tr) < 10:
        print(f"  [{group_name}] SKIP — only {len(X_tr)} train samples, too few to fit.")
        return None, {"group": group_name, "n_train": len(X_tr),
                      "n_test": len(X_te), "MAE": None, "RMSE": None, "R2": None}

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        verbose=verbose
    )

    y_pred = model.predict(X_te)
    mae    = mean_absolute_error(y_te, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
    r2     = r2_score(y_te, y_pred)

    print(f"  {group_name:<20} | n_train={len(X_tr):>4} | n_test={len(X_te):>4} "
          f"| best_iter={model.best_iteration:>4} "
          f"| MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}")

    return model, {
        "group"  : group_name,
        "n_train": len(X_tr),
        "n_test" : len(X_te),
        "MAE"    : round(mae,  4),
        "RMSE"   : round(rmse, 4),
        "R2"     : round(r2,   4),
    }

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. GROUP-WISE MODELS (age × gender)
#
#    Scaler + PCA already fit on the full train set.
#    Here we just slice the already-transformed matrices
#    by group — no re-fitting, no leakage.
# ═══════════════════════════════════════════════════════════
GROUPS = [
    ("Male_lt60",    {"gender": "Male",   "age_group": "lt60"}),
    ("Male_60to70",  {"gender": "Male",   "age_group": "60to70"}),
    ("Male_gt70",    {"gender": "Male",   "age_group": "gt70"}),
    ("Female_lt60",  {"gender": "Female", "age_group": "lt60"}),
    ("Female_60to70",{"gender": "Female", "age_group": "60to70"}),
    ("Female_gt70",  {"gender": "Female", "age_group": "gt70"}),
]

group_models  = {}
group_results = []

print("=" * 85)
print("Group-wise XGBoost (subject-aware split, PCA fit on train only)")
print("=" * 85)

for gname, filters in GROUPS:
    # Build boolean masks on original df → reuse transformed X_train / X_test
    tr_mask = (
        (train_df["gender"]    == filters["gender"]) &
        (train_df["age_group"] == filters["age_group"])
    ).values
    te_mask = (
        (test_df["gender"]    == filters["gender"]) &
        (test_df["age_group"] == filters["age_group"])
    ).values

    X_tr_g, y_tr_g = X_train[tr_mask], y_train[tr_mask]
    X_te_g, y_te_g = X_test[te_mask],  y_test[te_mask]

    model, metrics = train_xgb_group(X_tr_g, y_tr_g, X_te_g, y_te_g, gname)
    group_models[gname] = model
    group_results.append(metrics)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. GLOBAL MODEL (all groups, subject-aware split)
# ═══════════════════════════════════════════════════════════
print("=" * 85)
print("Global XGBoost (all data, subject-aware split)")
print("=" * 85)

model_global, metrics_global = train_xgb_group(
    X_train, y_train, X_test, y_test, "Global"
)
group_results.append(metrics_global)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. RESULTS SUMMARY TABLE
# ═══════════════════════════════════════════════════════════
results_df = pd.DataFrame(group_results)
results_df = results_df.sort_values("MAE", na_position="last")
print("\n", results_df.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. PERFORMANCE VISUALISATIONS
# ═══════════════════════════════════════════════════════════
plot_df = results_df[results_df["group"] != "Global"].dropna(subset=["MAE"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#EF5350", "#FF7043", "#42A5F5"]

for ax, metric, color in zip(axes, ["MAE", "RMSE", "R2"], colors):
    bars = ax.bar(plot_df["group"], plot_df[metric], color=color,
                  edgecolor="white", alpha=0.88)
    ax.set_title(f"{metric} by Group (subject-aware)", fontsize=12)
    ax.set_xlabel("Group")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=35)
    # Reference line from global model
    global_val = metrics_global.get(metric)
    if global_val is not None:
        ax.axhline(global_val, color="black", linestyle="--",
                   linewidth=1.2, label=f"Global ({global_val:.3f})")
        ax.legend(fontsize=8)

plt.suptitle("XGBoost Performance — Subject-Aware Split", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("xgb_group_performance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. GLOBAL MODEL — RESIDUAL & PREDICTED vs ACTUAL
# ═══════════════════════════════════════════════════════════
y_pred_global = model_global.predict(X_test)

meta_test = test_df[["subject#", "age_group", "gender"]].copy().reset_index(drop=True)
meta_test["y_true"]   = y_test
meta_test["y_pred"]   = y_pred_global
meta_test["residual"] = y_test - y_pred_global
meta_test["abs_err"]  = np.abs(meta_test["residual"])

age_palette = {"lt60": "#EF5350", "60to70": "#42A5F5", "gt70": "#26A69A"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
for grp, g in meta_test.groupby("age_group"):
    axes[0].scatter(g["y_true"], g["y_pred"],
                    alpha=0.4, s=18, label=grp, color=age_palette.get(grp, "gray"))
lim = [meta_test[["y_true","y_pred"]].min().min(),
       meta_test[["y_true","y_pred"]].max().max()]
axes[0].plot(lim, lim, "k--", linewidth=1.5, label="Perfect")
axes[0].set_xlabel("Actual total_UPDRS")
axes[0].set_ylabel("Predicted total_UPDRS")
axes[0].set_title("Predicted vs Actual (Global Model)")
axes[0].legend(title="Age Group", fontsize=8)

# Residuals vs Predicted
for grp, g in meta_test.groupby("age_group"):
    axes[1].scatter(g["y_pred"], g["residual"],
                    alpha=0.4, s=18, label=grp, color=age_palette.get(grp, "gray"))
axes[1].axhline(0, color="black", linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Predicted total_UPDRS")
axes[1].set_ylabel("Residual (actual − predicted)")
axes[1].set_title("Residuals vs Predicted")
axes[1].legend(title="Age Group", fontsize=8)

plt.tight_layout()
plt.savefig("xgb_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. ERROR DISTRIBUTION — Age group × Gender heatmap
# ═══════════════════════════════════════════════════════════
pivot = (meta_test
         .groupby(["age_group", "gender"])["abs_err"]
         .mean()
         .unstack())

plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Mean Absolute Error"})
plt.title("Mean Absolute Error — Age Group × Gender (Global Model)")
plt.tight_layout()
plt.savefig("xgb_error_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 13. FEATURE IMPORTANCE (Global Model)
# ═══════════════════════════════════════════════════════════
importances = pd.Series(
    model_global.feature_importances_,
    index=FEATURE_NAMES
).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
importances.plot(kind="bar", color="#42A5F5", edgecolor="white", alpha=0.9)
plt.title("XGBoost Feature Importance — Global Model (gain)")
plt.ylabel("Importance")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig("xgb_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print(importances)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 14. FINAL SUMMARY
# ═══════════════════════════════════════════════════════════
print("\n" + "═"*55)
print(" FINAL RESULTS SUMMARY (subject-aware split)")
print("═"*55)
print(results_df.to_string(index=False))
print("═"*55)
print(f"\nTest subjects ({len(test_subjects_all)}): {sorted(test_subjects_all)}")
print(f"Note: R² reflects honest generalisation to unseen patients.")
print(f"      Random-split R² would be ~0.93-0.97 (inflated by leakage).")